pip install -U sentence-transformers

In [1]:
import time
import math
import requests
from bs4 import BeautifulSoup
import pandas as pd
from collections import namedtuple
import re
#from sentence_transformers import SentenceTransformer, util
import time
from tqdm import tqdm
from collections import namedtuple
import random
import numpy as np

In [5]:
dfp = pd.read_csv('productos_ws.csv')

In [6]:
dfp.head(15)

,campaign_PK,product_PK,product_name,product_physicalreference,ordetail_created,Articulos_vendidos,buyers,visitas,visitors,precio_provedor,precio_privalia,CR,decil_visitors,decil_CR
0,341170,64161622,NAUTICA VOYAGE N-83 EAU DE TOILETTE 100 ML,938230-UNICO,27/10/2025,6,5,29,2,767.24,275.0012,2.500000,1,10
1,341170,64160797,MONTBLANC LEGEND BLUE EAU DE PARFUM 100 ML,1C7L027A01-UNICO,27/10/2025,2,2,13,1,2284.48,1137.9948,2.000000,1,10
2,341170,64159822,GUESS LOVE SEDUCTIVE RED FOR WOMEN BODY SPRAY ...,32243B-UNICO,27/10/2025,8,8,37,7,310.34,145.0000,1.142857,3,10
3,341170,64159624,PARIS HILTON CAN CAN EAU DE PARFUM 100 ML,151430376-UNICO,27/10/2025,1,1,5,1,1379.31,550.0024,1.000000,1,10
4,343681,64184602,SET PARIS HILTON GOLD RUSH,2001400-UNICO,27/10/2025,1,1,5,1,1336.21,1008.0052,1.000000,1,10
5,341170,64159813,GUESS LOVE NIRVANA DREAM BODY MIST 250 ML,32694-UNICO,27/10/2025,4,4,29,4,310.34,154.0016,1.000000,1,10
6,341170,64158631,GLORIA VANDERBILT VANDERBILT EAU DE PARFUM 100 ML,72001-UNICO,27/10/2025,1,1,14,1,387.93,201.0048,1.000000,1,10
7,342379,64183858,"JUEGO DE GORRO, BUFANDA Y GUANTES BLUELANDER",CBHH-0338-UNICO,27/10/2025,1,1,2,1,275.00,178.9996,1.000000,1,10
8,342619,64101175,PLAYERA CALVIN KLEIN JEANS CORAL MAXIMONOGRAMA,CJ4T5624-BNL,27/10/2025,1,1,8,1,947.41,449.0012,1.000000,1,10
9,341146,64084633,PERRY ELLIS 18 ORCHID EAU DE PARFUM 100 ML,04401276-UNICO,27/10/2025,1,1,5,1,1120.69,584.9996,1.000000,1,10


In [7]:
def limpia_precio(txt):
    if not txt: 
        return None
    # quita símbolos comunes de MercadoLibre (puntos de miles, comas, $)
    t = txt.replace("\xa0", " ").replace(".", "").replace(",", "").replace("$", "").strip()
    # a veces viene junto con centavos en otro span; nos quedamos con entero
    try:
        return int("".join(ch for ch in t if ch.isdigit()))
    except ValueError:
        return None
def parse_item(li):
    # Títulos
    nombre = None
    brand_span = li.select_one(".poly-component__brand")
    if brand_span:
        nombre = brand_span.get_text(strip=True)
        
    descripcion = None
    t = li.find("h2")
    if t:
        descripcion = t.get_text(strip=True)
    else:
        t = li.find("h3")
        if t: descripcion = t.get_text(strip=True)

    # Link
    a = li.find("a", href=True)
    link = a["href"] if a else None

    # Imagen
    img = li.find("img")
    imagen = img.get("data-src") or img.get("src") if img else None

    # Precios (ML usa varias clases; probamos alternativas)
    precio_actual = None
    precio_antes = None

    # Número entero del precio actual
    actual_span = li.select_one(".poly-price__current .andes-money-amount__fraction")
    if actual_span:
        precio_actual = limpia_precio(actual_span.get_text())

    # Posible precio anterior (tachado) aparece con otra clase o data-test-id
    antes = (li.select_one(".andes-money-amount.comparative-price .andes-money-amount__fraction")
             or li.select_one("[data-test-id='price-before'] .andes-money-amount__fraction")
             or li.select_one(".ui-search-price__second-line .price-tag__disabled .price-tag-amount"))
    if antes:
        precio_antes = limpia_precio(antes.get_text())

    return Articulo(nombre, descripcion,  precio_actual, link, imagen)

Articulo = namedtuple("Articulo", ["nombre", "descripcion",  "precio_actual", "link", "imagen"])
articulos = []

# ---------- Config ----------
PAIS = "com.mx"  # cambia a "com.co", "com.ar", etc.

PAGINAS = 1            # cuántas páginas raspar
ESPERA_S = 30         # pausa entre páginas (ser buen ciudadano)
BASE = f"https://listado.mercadolibre.{PAIS}"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "es-MX,es;q=0.9"
}
# ---------- Scrape con paginado ----------
def scrap_page(PAGINAS,URL,HEADERS):
    for page in range(1, PAGINAS + 1):
        # En ML el paginado suele ser con _Desde_ o _Desde_XX; esta variante funciona en varios países:
        sep = "" if URL.endswith("/") else "/"
        url_pag = URL if page == 1 else f"{URL}{sep}_Desde_{(page-1)*48+1}"
        try:
            resp = requests.get(url_pag, headers=HEADERS, timeout=20)
            resp.raise_for_status()
        except requests.HTTPError as e:
            print(f"[WARN] {e} -> {url_pag}")
            break

        soup = BeautifulSoup(resp.text, "html.parser")

        # Cada item suele estar en <li> del layout de búsqueda
        items = soup.select("li.ui-search-layout__item, li.ui-search-layout__item--stack")
        if not items:
            # fallback genérico
            items = soup.select("li.ui-search-result__wrapper")
        if not items:
            print(f"[WARN] No se hallaron items en página {page}.")
            break

        for li in items:
            try:
                articulos.append(parse_item(li))
            except Exception as exc:
                # si un item rompe, sigue con el siguiente
                print(f"[WARN] item con error: {exc}")

        time.sleep(ESPERA_S)
        df = pd.DataFrame(articulos)
        return df










In [ ]:
# Modelo de embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")

QUERY = "CAFETERA-NEGRA-CFN601-NINJA"
URL = f"{BASE}/{QUERY}"
df_scrap
df_scrap=scrap_page(PAGINAS,URL,HEADERS)[["nombre", "descripcion", "precio_actual"]]
df_scrap

df_scrap["titulo"] = df_scrap["descripcion"].str.split(",").str[0]
df_scrap["embedding"] = df_scrap["titulo"].apply(lambda x: model.encode(str(x), convert_to_tensor=True))
    
# Calcular similitud coseno
query_emb = model.encode(QUERY, convert_to_tensor=True)
df_scrap["similaridad"] = df_scrap["embedding"].apply(lambda emb: util.cos_sim(query_emb, emb).item())

df_scrap

In [ ]:
list(dfp.iloc[0,:])

 df_scrap=scrap_page(PAGINAS,URL,HEADERS)[["nombre", "descripcion", "precio_actual"]]

In [9]:
columnas = ["pagina","product_PK", "product_name", "campaign_PK", "precio_privalia", "n_productos","precio_min", "precio_max", "precio_avg", "link_min"]
df = pd.DataFrame(columns=columnas)
inicio = time.time()
for i in range(4):
    
    print(f'ESCRAPEANDO EL PRODUCTO NÚMERO {i}')
    p = list(dfp.iloc[i,:])
    print(list(dfp.product_name)[i])
    print(f'con un precio de {list(dfp.precio_privalia)[i]}')
    QUERY = list(dfp.product_name)[i]  # tu búsqueda
    URL = f"{BASE}/{QUERY.replace(' ', '-')}"
    articulos = []
    print(URL)
    df_scrap=scrap_page(PAGINAS,URL,HEADERS)#[["nombre", "descripcion", "precio_actual"]]
    print (f'se escrapearon {df_scrap.shape[0]} articulos')
    df_scrap["titulo"] = df_scrap["descripcion"].str.split(",").str[0]
    # Obtener embeddings para todos los nombres
    #df_scrap["embedding"] = df_scrap["titulo"].apply(lambda x: model.encode(str(x), convert_to_tensor=True))
    # Calcular similitud coseno
    #query_emb = model.encode(QUERY, convert_to_tensor=True)
    #df_scrap["similaridad"] = df_scrap["embedding"].apply(lambda emb: util.cos_sim(query_emb, emb).item())
    p_min =  list(dfp.precio_privalia)[i]*0.8
    p_max = list(dfp.precio_privalia)[i]*1.2
    #df_sal = df_scrap[(df_scrap.precio_actual> p_min) & (df_scrap.precio_actual< p_max) &(df_scrap.similaridad>=0.7)]
    df_sal = df_scrap.copy()
    print(f'columnas: {df_sal.columns}')
    n_productos = df_sal.shape[0]
    print(f'quedan {n_productos} productos filtrados')
    min_p = df_sal.precio_actual.min()
    print(f'el precio mínimo es: {min_p}')
    max_p = df_sal.precio_actual.max()
    avg = df_sal.precio_actual.mean()
    if n_productos != 0:
        #link_page1 = df_sal[df_sal.precio_actual == min_p].link
        link_min = list(df_sal[df_sal.precio_actual == min_p].link)[0]

    else: 
        link_min = np.nan
        
    
    df.loc[len(df)] = ["MeLi", list(dfp.product_PK)[i], list(dfp.product_name)[i], list(dfp.campaign_PK)[i], list(dfp.precio_privalia)[i],df_sal.shape[0], min_p, max_p, avg,link_min ]
    espera=random.normalvariate(30, 10)
    print(f'segundos de sespera {espera}')
    time.sleep(abs(espera))
fin = time.time()
duracion = fin - inicio
print(f"Duración: {fin - inicio:.4f} segundos")

ESCRAPEANDO EL PRODUCTO NÚMERO 0
NAUTICA VOYAGE N-83 EAU DE TOILETTE 100 ML
con un precio de 275.0012
https://listado.mercadolibre.com.mx/NAUTICA-VOYAGE-N-83-EAU-DE-TOILETTE-100-ML
se escrapearon 52 articulos
columnas: Index(['nombre', 'descripcion', 'precio_actual', 'link', 'imagen', 'titulo'], dtype='str')
quedan 52 productos filtrados
el precio mínimo es: 284
segundos de sespera 13.151965374317925
ESCRAPEANDO EL PRODUCTO NÚMERO 1
MONTBLANC LEGEND BLUE  EAU DE PARFUM 100 ML
con un precio de 1137.9948
https://listado.mercadolibre.com.mx/MONTBLANC-LEGEND-BLUE--EAU-DE-PARFUM-100-ML
se escrapearon 34 articulos
columnas: Index(['nombre', 'descripcion', 'precio_actual', 'link', 'imagen', 'titulo'], dtype='str')
quedan 34 productos filtrados
el precio mínimo es: 740
segundos de sespera 27.084131434750237
ESCRAPEANDO EL PRODUCTO NÚMERO 2
GUESS LOVE SEDUCTIVE RED FOR WOMEN BODY SPRAY 250 ML
con un precio de 145.0
https://listado.mercadolibre.com.mx/GUESS-LOVE-SEDUCTIVE-RED-FOR-WOMEN-BODY-SPRA

for i in range(5):  # puedes cambiar 5 por el número de repeticiones que quieras
    valor = random.normalvariate(15, 5)
    print(valor)
    time.sleep(10)  # espera 10 segundos antes de continuar

In [10]:
df.shape

(4, 10)

In [11]:
df.head(30)

,pagina,product_PK,product_name,campaign_PK,precio_privalia,n_productos,precio_min,precio_max,precio_avg,link_min
0,MeLi,64161622,NAUTICA VOYAGE N-83 EAU DE TOILETTE 100 ML,341170,275.0012,52,284,1439,495.500000,https://www.mercadolibre.com.mx/perfume-nautic...
1,MeLi,64160797,MONTBLANC LEGEND BLUE EAU DE PARFUM 100 ML,341170,1137.9948,34,740,2929,1382.441176,https://www.mercadolibre.com.mx/explorer-ultra...
2,MeLi,64159822,GUESS LOVE SEDUCTIVE RED FOR WOMEN BODY SPRAY ...,341170,145.0000,8,200,640,332.625000,https://www.mercadolibre.com.mx/guess-seductiv...
3,MeLi,64159624,PARIS HILTON CAN CAN EAU DE PARFUM 100 ML,341170,550.0024,48,406,1955,970.770833,https://www.mercadolibre.com.mx/paris-hilton-e...


In [ ]:
df.to_csv("Web_Scraping.csv",index=False, encoding="utf-8-sig")

In [ ]:
list(df_sal[df_sal.precio_actual == min_p].link)[0]

In [ ]:
df